# DOF Coordinate Extraction -- nx=24, ny=60, Lx=0.2mm baseline mesh

**Purpose only:** recover the physical (x, y) coordinates of every degree of freedom in the
P1 Lagrange scalar space `V`, for the exact same mesh used throughout this project's
Lx=0.2mm, tau=1.1 notebooks (baseline mesh, nx=24, ny=60 -- the mesh used by
`..._MixedDirection_Fields.ipynb` / the `eigcheck_theta*.npz` files, 25x61=1525 DOFs).

This notebook does **no physics solve** -- it only builds the mesh and function space and
calls `V.tabulate_dof_coordinates()`. It is independent of, and does not touch, any other
running Continuation notebook or its Drive backups. Runtime: well under a minute.

Output: `dof_coords_nx24_ny60_Lx0.2mm.npz` with arrays `x`, `y` (each shape (1525,)), in the
same DOF order as `s_field`/`Cw`/`CgO2`/`p` inside `eigcheck_theta*.npz` -- i.e.
`x[i], y[i]` is the physical location of `s_field[i]`.

After running, download the output file (last cell does this automatically) and share it
back so Figure 8 (`mixed_direction_field_comparison.png`) can be redrawn with contour lines
matching Figure 1's style.


## Step 0 -- install DOLFINx (same setup as every other notebook in this project)

In [ ]:
!git clone -q https://github.com/seoultechpse/fenicsx-colab.git
%run fenicsx-colab/setup_fenicsx.py --version 0.11


## Step 1 -- build the exact mesh/function space and tabulate DOF coordinates

In [ ]:
%%fenicsx -np 1

import numpy as np
from mpi4py import MPI
from dolfinx import mesh, fem

# Exact constants from pemfc_lib.py for the Lx=0.2mm project notebooks
# (w_ch, w_rb are the DEFAULT_PARAMS values; Lx is this project's override, not the
# DEFAULT_PARAMS default of 0.30e-3; nx, ny are the baseline mesh used for the
# mixed-direction / eigcheck states -- NOT the RandomDirectionScreen coarse (12,30) or
# fine (36,90) MESH_SPECS).
w_ch, w_rb = 0.8e-3, 1.2e-3
Lx = 0.2e-3
Ly = w_ch + w_rb          # = 2.0e-3 m
nx, ny = 24, 60

domain = mesh.create_rectangle(
    MPI.COMM_WORLD, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle
)
V = fem.functionspace(domain, ("Lagrange", 1))
coords = V.tabulate_dof_coordinates()   # shape (num_dofs, 3), columns are (x, y, z)

print("num DOFs:", coords.shape[0], " (expect 1525 = 25*61)")
print("x range:", coords[:, 0].min(), coords[:, 0].max(), " (expect 0 .. %.6g)" % Lx)
print("y range:", coords[:, 1].min(), coords[:, 1].max(), " (expect 0 .. %.6g)" % Ly)

np.savez(
    "/content/dof_coords_nx24_ny60_Lx0.2mm.npz",
    x=coords[:, 0], y=coords[:, 1],
)
print("saved /content/dof_coords_nx24_ny60_Lx0.2mm.npz")


## Step 2 -- download the output file

In [ ]:
from google.colab import files
files.download("/content/dof_coords_nx24_ny60_Lx0.2mm.npz")
